# Social Content Mix Efficiency

## 1) Problem framing
**Business question:** Which content and channel choices maximize donation value and referral conversion per post?

- Explanatory goal: quantify associations between content attributes and donation value.
- Predictive goal: predict whether a post lands in the top referral quartile before publishing.
- Success metrics: explanatory `R2` and coefficient direction stability; predictive `ROC-AUC`, `F1`, and recall.

## 2) Data acquisition and preparation
- Tables: `social_media_posts`, `donations`.
- Reproducible prep: `ColumnTransformer` + `Pipeline`.

## 3) Exploration
- Distribution checks, missingness, and segment summaries.

## 4) Modeling
- Explanatory: linear regression (log donation value).
- Predictive: random forest classifier (high-referral class).

## 5) Evaluation and selection
- Baseline vs candidate models, holdout metrics, and CV.

## 6) Feature selection
- Coefficients and permutation importance.

## 7) Deployment
- Endpoint candidate: `/api/reports/trends/social-content-mix`.
- UI candidate: Reports strategy card + pre-publish advisor.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

posts = load_table('social_media_posts')
donations = load_table('donations')
donations = donations[donations['referral_post_id'].notna()].copy()
donations['referral_post_id'] = donations['referral_post_id'].astype(int)
donation_rollup = donations.groupby('referral_post_id').agg(
    referred_count=('donation_id', 'count'),
    referred_amount=('amount', 'sum')
).reset_index()
df = posts.merge(donation_rollup, left_on='post_id', right_on='referral_post_id', how='left')
df[['referred_count', 'referred_amount']] = df[['referred_count', 'referred_amount']].fillna(0)
df['target_value'] = np.log1p(df['estimated_donation_value_php'].clip(lower=0))
df['high_referral'] = (df['donation_referrals'] >= df['donation_referrals'].quantile(0.75)).astype(int)

feature_cols = [
    'platform', 'post_type', 'media_type', 'sentiment_tone', 'content_topic',
    'has_call_to_action', 'features_resident_story', 'is_boosted', 'post_hour',
    'num_hashtags', 'caption_length', 'engagement_rate', 'impressions', 'reach'
]
X = df[feature_cols].copy()
y_reg = df['target_value']
y_clf = df['high_referral']

cat_cols = [c for c in feature_cols if X[c].dtype == 'object' or X[c].dtype == 'bool']
num_cols = [c for c in feature_cols if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

Xtr, Xte, ytr_reg, yte_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory model R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory model MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

Xtrc, Xtec, ytrc, ytec = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
rf = Pipeline([('prep', prep), ('model', RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=4))])
baseline.fit(Xtrc, ytrc)
rf.fit(Xtrc, ytrc)
proba_base = baseline.predict_proba(Xtec)[:, 1]
proba_rf = rf.predict_proba(Xtec)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)
print('Predictive baseline AUC:', round(roc_auc_score(ytec, proba_base), 3))
print('Predictive RF AUC:', round(roc_auc_score(ytec, proba_rf), 3))
print('Predictive RF F1:', round(f1_score(ytec, pred_rf), 3))

cv = cross_validate(rf, X, y_clf, cv=5, scoring=['roc_auc', 'f1'])
print('CV ROC-AUC mean:', round(float(cv['test_roc_auc'].mean()), 3), 'std:', round(float(cv['test_roc_auc'].std()), 3))

perm = permutation_importance(rf, Xtec, ytec, n_repeats=8, random_state=42, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
print('\nTop predictive features:')
print(imp.round(4).to_string())

lin_feature_names = lin.named_steps['prep'].get_feature_names_out()
coef_values = np.ravel(lin.named_steps['model'].coef_)
usable = min(len(coef_values), len(lin_feature_names))
coef = pd.Series(coef_values[:usable], index=lin_feature_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
print('\nTop explanatory coefficients:')
print(coef.round(4).to_string())

print('\nDecision recommendations: prioritize high-impact content mixes and validate with A/B scheduling experiments.')


C:\Users\ethan\AppData\Local\Temp\ipykernel_35240\2661784589.py:25: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  donations['referral_post_id'] = donations['referral_post_id'].astype(int)
C:\Users\ethan\AppData\Local\Temp\ipykernel_35240\266

Explanatory model R2: 0.7
Explanatory model MAE: 2.128


Predictive baseline AUC: 0.5
Predictive RF AUC: 0.986
Predictive RF F1: 0.861


CV ROC-AUC mean: 0.974 std: 0.01



Top predictive features:
post_type                  0.0553
features_resident_story    0.0526
engagement_rate            0.0239
impressions                0.0129
reach                      0.0119
caption_length             0.0103
post_hour                  0.0077
platform                   0.0033
has_call_to_action         0.0022
sentiment_tone             0.0018

Top explanatory coefficients:
cat__post_type_ImpactStory            2.8832
cat__post_type_EventPromotion        -2.6190
cat__post_type_ThankYou              -2.4085
cat__post_type_EducationalContent    -2.2541
cat__post_type_FundraisingAppeal      2.2155
cat__post_type_Campaign               2.1829
num__engagement_rate                  1.8243
cat__platform_WhatsApp                1.4709
cat__features_resident_story_False   -0.8943
cat__features_resident_story_True     0.8943

Decision recommendations: prioritize high-impact content mixes and validate with A/B scheduling experiments.


In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if ytec.nunique() >= 2:
    proba = rf.predict_proba(Xtec)[:, 1]
    print_calibration_bins(ytec.values, proba)
    print_threshold_scan(ytec.values, proba)
    fairness_binary(rf, Xtec, ytec, df.loc[Xtec.index], ['platform'], min_n=15)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)
